# Unknown-variance Gaussian changes: Section 5.1 UI detector

This notebook generates a replacement for the Gaussian table in Section 8.3 where the composite Panel B uses the Section 5.1 universal-inference t detector

\[
D_t^{t,w}=\inf_{\theta\in\mathbb R}\sum_{s=1}^t w_s R_{s:t}^{\theta},
\qquad
\tau_A^{t,w}=\inf\{t: D_t^{t,w}\ge A\}.
\]

The notebook is self-contained for Google Colab. It does not depend on the GitHub repository at runtime. Set `QUICK_RUN = True` for a smoke test; leave it `False` for the paper-scale simulation settings copied from the previous Gaussian scripts.

In [ ]:
# Main switches.
QUICK_RUN = False          # True = small smoke test; False = paper-scale defaults.
RUN_PANEL_A = True
RUN_PANEL_B = True
INCLUDE_REDUCED_G_POINT_NULL = True  # Panel B is UI either way; this only controls Panel A's comparison column.

# Random seeds, matching the style of the original scripts.
POINT_SEED = 20260730
ARL_SEED = 20260731
PFA_SEED = 20260801

# Panel A thresholds.
POINT_LEVELS = [2e4, 1e8, 1e20, 1e40]
POINT_REPS = 2500 if not QUICK_RUN else 250

# Panel B controls. These mirror the previous reduced-G table's replication counts.
ARL_A = 1e12
ARL_GRID = [(15, 100), (30, 80), (60, 60), (120, 40)]
PFA_ALPHA = 1e-8
PFA_GRID = [(15, 100), (30, 80), (60, 60)]

if QUICK_RUN:
    ARL_GRID = [(15, 8), (30, 6)]
    PFA_GRID = [(15, 8)]

# Horizon padding for post-change simulation. Increase if detection_fraction is below 1.
POST_MULTIPLIER = 3.5
POST_PAD = 160

# Section 5.1 regularized online predictors in equation (22): m0=0, v0=2, nu0=2.
M0 = 0.0
V0 = 2.0
NU0 = 2.0

In [ ]:
import importlib.util
import math
import subprocess
import sys
from pathlib import Path


def ensure_package(package):
    if importlib.util.find_spec(package) is not None:
        return
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    except subprocess.CalledProcessError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--user", package])
    if importlib.util.find_spec(package) is None:
        raise RuntimeError(f"Package {package!r} could not be imported after installation.")


for package in ["numba", "pandas"]:
    ensure_package(package)

import numpy as np
import pandas as pd
from numba import njit, prange

I_T = 0.5 * math.log(2.0)
print(f"I_t = {I_T:.12f}")

In [ ]:
def log_pi_weight_py(s: int) -> float:
    # pi_s = 0.5 / {s log(e s)^2}; the sum is <= 1.
    return math.log(0.5) - math.log(s) - 2.0 * math.log(math.log(math.e * s))


def sample_gauss(rng, reps, horizon, change, a, u, b, v):
    """Rows are independent paths. Change occurs after `change` observations."""
    out = np.empty((reps, horizon), dtype=np.float64)
    if change:
        out[:, :change] = rng.normal(a, math.sqrt(u), size=(reps, change))
    if horizon > change:
        out[:, change:] = rng.normal(b, math.sqrt(v), size=(reps, horizon - change))
    return out


def first_crossing(logpaths, boundary):
    hit = logpaths >= boundary
    any_hit = hit.any(axis=1)
    out = np.full(logpaths.shape[0], logpaths.shape[1] + 1, dtype=int)
    out[any_hit] = hit[any_hit].argmax(axis=1) + 1
    return out


def gaussian_oracle_paths(x, theta=0.0, c=1.0, nullvar=1.0):
    """One-start UI-t, reduced-G, and full-Gaussian predictive log e-processes."""
    reps, horizon = x.shape
    ui = np.empty((reps, horizon), dtype=np.float64)
    gg = np.empty((reps, horizon), dtype=np.float64)
    full = np.empty((reps, horizon), dtype=np.float64)
    means = np.zeros(reps)
    m2 = np.zeros(reps)
    pred_ui = np.zeros(reps)
    pred_full = np.zeros(reps)
    sums = np.zeros(reps)
    sumsq = np.zeros(reps)
    null_log = np.zeros(reps)
    for t in range(horizon):
        z = x[:, t]
        pm = np.zeros(reps) if t == 0 else means.copy()
        pv = (2.0 + m2) / (t + 2.0)
        pred_ui += -0.5 * np.log(pv) - (z - pm) ** 2 / (2.0 * pv)
        pred_full += -0.5 * np.log(2.0 * np.pi * pv) - (z - pm) ** 2 / (2.0 * pv)
        null_log += -0.5 * math.log(2.0 * np.pi * nullvar) - (z - theta) ** 2 / (2.0 * nullvar)
        n = t + 1
        delta = z - means
        means += delta / n
        m2 += delta * (z - means)
        sums += z
        sumsq += z * z
        rss = np.maximum(sumsq - 2.0 * theta * sums + n * theta * theta, 1e-300)
        ui[:, t] = pred_ui + 0.5 * n * (1.0 + np.log(rss / n))
        S = sums - n * theta
        den = np.maximum((n + c * c) * rss - S * S, 1e-300)
        gg[:, t] = 0.5 * (math.log(c * c) - math.log(n + c * c)) + 0.5 * n * (
            np.log((n + c * c) * rss) - np.log(den)
        )
        full[:, t] = pred_full - null_log
    return ui, gg, full

In [ ]:
def J(mean, theta, variance=1.0):
    """Section 5.1 t-information J_{mean,variance}(theta)."""
    return 0.5 * math.log1p((mean - theta) ** 2 / variance)


def logaddexp(a, b):
    m = max(a, b)
    return m + math.log(math.exp(a - m) + math.exp(b - m))


def golden_minimize_on_unit_interval(fn, iters=96):
    """Small dependency-free convex minimization over theta in [0, 1]."""
    lo, hi = 0.0, 1.0
    gr = (math.sqrt(5.0) - 1.0) / 2.0
    c = hi - gr * (hi - lo)
    d = lo + gr * (hi - lo)
    fc = fn(c)
    fd = fn(d)
    for _ in range(iters):
        if fc <= fd:
            hi = d
            d = c
            fd = fc
            c = hi - gr * (hi - lo)
            fc = fn(c)
        else:
            lo = c
            c = d
            fc = fd
            d = lo + gr * (hi - lo)
            fd = fn(d)
    theta = 0.5 * (lo + hi)
    return theta, fn(theta)


def finite_logsum_value(T, d, logw_prefix, logw_post):
    """Two-block finite-prefix log-sum balance for the UI sum detector."""
    def objective(theta):
        prefix = logw_prefix + T * J(0.0, theta)
        post = logw_post + d * J(1.0, theta)
        return logaddexp(prefix, post)
    return golden_minimize_on_unit_interval(objective)[1]


def finite_prefix_prediction_ui(T, threshold, logw_prefix=0.0, logw_post=0.0):
    """Solve min_theta log(exp(logw1+T J0)+exp(logwpost+d J1)) = threshold."""
    lo = 0.0
    hi = max(10.0, 2.0 * (threshold - logw_post) / I_T)
    while finite_logsum_value(T, hi, logw_prefix, logw_post) < threshold:
        hi *= 2.0
    for _ in range(80):
        mid = 0.5 * (lo + hi)
        if finite_logsum_value(T, mid, logw_prefix, logw_post) < threshold:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)


def solve_T_for_pfa(c, alpha=PFA_ALPHA):
    """Solve T / d0(T) = c with d0(T) = {log(1/alpha)-log pi_{T+1}} / I_t."""
    L = math.log(1.0 / alpha)
    T = 1000.0
    for _ in range(250):
        effective = L - log_pi_weight_py(int(round(T)) + 1)
        new_T = c * effective / I_T
        if abs(new_T - T) < 1e-8:
            break
        T = 0.5 * T + 0.5 * new_T
    Ti = int(round(T))
    d0 = (L - log_pi_weight_py(Ti + 1)) / I_T
    return Ti, d0

In [ ]:
@njit
def log_pi_weight_numba(s):
    return math.log(0.5) - math.log(s) - 2.0 * math.log(math.log(math.e * s))


@njit
def ui_logD_and_derivative(n_arr, sum_arr, sumsq_arr, pred_arr, logw_arr, active, theta):
    maxlog = -1.0e300
    for i in range(active):
        n = n_arr[i]
        rss = sumsq_arr[i] - 2.0 * theta * sum_arr[i] + n * theta * theta
        if rss < 1.0e-300:
            rss = 1.0e-300
        val = logw_arr[i] + pred_arr[i] + 0.5 * n * (1.0 + math.log(rss / n))
        if val > maxlog:
            maxlog = val

    weight_sum = 0.0
    deriv_sum = 0.0
    for i in range(active):
        n = n_arr[i]
        rss = sumsq_arr[i] - 2.0 * theta * sum_arr[i] + n * theta * theta
        if rss < 1.0e-300:
            rss = 1.0e-300
        val = logw_arr[i] + pred_arr[i] + 0.5 * n * (1.0 + math.log(rss / n))
        w = math.exp(val - maxlog)
        deriv = n * (n * theta - sum_arr[i]) / rss
        weight_sum += w
        deriv_sum += w * deriv
    return maxlog + math.log(weight_sum), deriv_sum / weight_sum


@njit
def ui_min_logD(n_arr, sum_arr, sumsq_arr, pred_arr, logw_arr, active):
    lo = 1.0e300
    hi = -1.0e300
    for i in range(active):
        mean = sum_arr[i] / n_arr[i]
        if mean < lo:
            lo = mean
        if mean > hi:
            hi = mean

    if hi - lo < 1.0e-12:
        val, _ = ui_logD_and_derivative(n_arr, sum_arr, sumsq_arr, pred_arr, logw_arr, active, lo)
        return val

    _, flo = ui_logD_and_derivative(n_arr, sum_arr, sumsq_arr, pred_arr, logw_arr, active, lo)
    if flo >= 0.0:
        val, _ = ui_logD_and_derivative(n_arr, sum_arr, sumsq_arr, pred_arr, logw_arr, active, lo)
        return val

    _, fhi = ui_logD_and_derivative(n_arr, sum_arr, sumsq_arr, pred_arr, logw_arr, active, hi)
    if fhi <= 0.0:
        val, _ = ui_logD_and_derivative(n_arr, sum_arr, sumsq_arr, pred_arr, logw_arr, active, hi)
        return val

    left = lo
    right = hi
    for _ in range(44):
        mid = 0.5 * (left + right)
        _, fmid = ui_logD_and_derivative(n_arr, sum_arr, sumsq_arr, pred_arr, logw_arr, active, mid)
        if fmid < 0.0:
            left = mid
        else:
            right = mid
    theta = 0.5 * (left + right)
    val, _ = ui_logD_and_derivative(n_arr, sum_arr, sumsq_arr, pred_arr, logw_arr, active, theta)
    return val


@njit
def ui_studentized_stop_one(x, pfa, threshold, check_start, m0, v0, nu0):
    H = x.size
    n_arr = np.zeros(H, dtype=np.int64)
    sum_arr = np.zeros(H, dtype=np.float64)
    sumsq_arr = np.zeros(H, dtype=np.float64)
    mean_arr = np.zeros(H, dtype=np.float64)
    m2_arr = np.zeros(H, dtype=np.float64)
    pred_arr = np.zeros(H, dtype=np.float64)
    logw_arr = np.zeros(H, dtype=np.float64)

    for t in range(H):
        active = t + 1
        z = x[t]
        logw_arr[t] = log_pi_weight_numba(t + 1) if pfa else 0.0

        for s in range(active):
            k = n_arr[s]
            pm = m0 if k == 0 else mean_arr[s]
            pv = (v0 + m2_arr[s]) / (k + nu0)
            if pv < 1.0e-300:
                pv = 1.0e-300
            pred_arr[s] += -0.5 * math.log(pv) - (z - pm) * (z - pm) / (2.0 * pv)

            new_n = k + 1
            delta = z - mean_arr[s]
            mean_arr[s] += delta / new_n
            m2_arr[s] += delta * (z - mean_arr[s])
            n_arr[s] = new_n
            sum_arr[s] += z
            sumsq_arr[s] += z * z

        if t + 1 > check_start:
            if ui_min_logD(n_arr, sum_arr, sumsq_arr, pred_arr, logw_arr, active) >= threshold:
                return t + 1
    return H + 1


@njit(parallel=True)
def ui_studentized_stops(x, pfa, threshold, check_start, m0, v0, nu0):
    out = np.empty(x.shape[0], dtype=np.int64)
    for r in prange(x.shape[0]):
        out[r] = ui_studentized_stop_one(x[r], pfa, threshold, check_start, m0, v0, nu0)
    return out


# Tiny compilation call. This keeps the first real row from paying the compile cost.
_compile_x = np.zeros((1, 3), dtype=np.float64)
_ = ui_studentized_stops(_compile_x, False, 10.0, 0, M0, V0, NU0)
print("Compiled Section 5.1 UI detector.")

In [ ]:
def summarize_detection(stops, horizon, change):
    detected = stops <= horizon
    delays = stops[detected] - change
    if delays.size == 0:
        return {
            "detection_fraction": float(detected.mean()),
            "median": float("nan"),
            "q10": float("nan"),
            "q90": float("nan"),
        }
    return {
        "detection_fraction": float(detected.mean()),
        "median": float(np.median(delays)),
        "q10": float(np.quantile(delays, 0.1)),
        "q90": float(np.quantile(delays, 0.9)),
    }


def run_panel_a():
    rng = np.random.default_rng(POINT_SEED)
    Bmax = math.log(2.0 * max(POINT_LEVELS))
    horizon = int(math.ceil(1.6 * (Bmax + 12.0) / I_T))
    x = sample_gauss(rng, POINT_REPS, horizon, 0, 0.0, 1.0, 1.0, 1.0)
    ui, reduced_g, _ = gaussian_oracle_paths(x, theta=0.0, c=1.0, nullvar=1.0)

    rows = []
    for A in POINT_LEVELS:
        specs = [("UI-t", ui, math.log(A))]
        if INCLUDE_REDUCED_G_POINT_NULL:
            specs.append(("reduced-G", reduced_g, math.log(2.0 * A)))
        for method, paths, boundary in specs:
            stops = first_crossing(paths, boundary)
            detected = stops <= horizon
            d = stops[detected]
            pred = boundary / I_T
            rows.append({
                "panel": "point-null",
                "A": A,
                "method": method,
                "reps": POINT_REPS,
                "horizon": horizon,
                "information": I_T,
                "boundary": boundary,
                "prediction": pred,
                "detection_fraction": float(detected.mean()),
                "median": float(np.median(d)) if d.size else float("nan"),
                "q10": float(np.quantile(d, 0.1)) if d.size else float("nan"),
                "q90": float(np.quantile(d, 0.9)) if d.size else float("nan"),
                "ratio": float(np.median(d) / pred) if d.size else float("nan"),
            })
    return pd.DataFrame(rows)


if RUN_PANEL_A:
    point_df = run_panel_a()
    display(point_df)
else:
    point_df = pd.DataFrame()

In [ ]:
def run_composite_ui_panel_b():
    rows = []

    rng = np.random.default_rng(ARL_SEED)
    L_arl = math.log(ARL_A)
    d0_arl = L_arl / I_T
    for c, reps in ARL_GRID:
        T = int(round(c * d0_arl))
        dft = finite_prefix_prediction_ui(T, L_arl, 0.0, 0.0)
        post = int(math.ceil(POST_MULTIPLIER * dft + POST_PAD))
        horizon = T + post
        x = sample_gauss(rng, reps, horizon, T, 0.0, 1.0, 1.0, 1.0)
        stops = ui_studentized_stops(x, False, L_arl, T, M0, V0, NU0)
        summary = summarize_detection(stops, horizon, T)
        row = {
            "panel": "composite-ui",
            "control": "ARL",
            "A": ARL_A,
            "alpha": np.nan,
            "T_over_d0": c,
            "T": T,
            "reps": reps,
            "information": I_T,
            "boundary": L_arl,
            "effective_boundary": L_arl,
            "d0": d0_arl,
            "d_ui_ft": dft,
            "horizon": horizon,
            **summary,
        }
        row["ratio_first"] = row["median"] / row["d0"]
        row["ratio_finite"] = row["median"] / row["d_ui_ft"]
        rows.append(row)
        print(row, flush=True)

    rng = np.random.default_rng(PFA_SEED)
    L_pfa = math.log(1.0 / PFA_ALPHA)
    for c, reps in PFA_GRID:
        T, d0 = solve_T_for_pfa(c, PFA_ALPHA)
        logw_prefix = log_pi_weight_py(1)
        logw_post = log_pi_weight_py(T + 1)
        dft = finite_prefix_prediction_ui(T, L_pfa, logw_prefix, logw_post)
        post = int(math.ceil(POST_MULTIPLIER * dft + POST_PAD))
        horizon = T + post
        x = sample_gauss(rng, reps, horizon, T, 0.0, 1.0, 1.0, 1.0)
        stops = ui_studentized_stops(x, True, L_pfa, T, M0, V0, NU0)
        summary = summarize_detection(stops, horizon, T)
        row = {
            "panel": "composite-ui",
            "control": "PFA",
            "A": np.nan,
            "alpha": PFA_ALPHA,
            "T_over_d0": c,
            "T": T,
            "reps": reps,
            "information": I_T,
            "boundary": L_pfa,
            "effective_boundary": L_pfa - logw_post,
            "d0": d0,
            "d_ui_ft": dft,
            "horizon": horizon,
            **summary,
        }
        row["ratio_first"] = row["median"] / row["d0"]
        row["ratio_finite"] = row["median"] / row["d_ui_ft"]
        rows.append(row)
        print(row, flush=True)

    return pd.DataFrame(rows)


if RUN_PANEL_B:
    composite_df = run_composite_ui_panel_b()
    display(composite_df)
else:
    composite_df = pd.DataFrame()

In [ ]:
def fmt_num(x, decimals=1):
    if not np.isfinite(x):
        return "--"
    if abs(x - round(x)) < 1e-9:
        return f"{int(round(x))}"
    return f"{x:.{decimals}f}"


def fmt_A_latex(A):
    if abs(A - 2e4) / A < 1e-12:
        return r"$2\times10^{4}$"
    exponent = int(round(math.log10(A)))
    if abs(A - 10 ** exponent) / A < 1e-12:
        return rf"$10^{{{exponent}}}$"
    return f"${A:.3g}$"


def cell_median_ratio(median, ratio):
    return rf"${fmt_num(median)}\;({ratio:.3f})$"


def make_latex_table(point_df, composite_df):
    include_reduced = INCLUDE_REDUCED_G_POINT_NULL and not point_df.empty and (point_df["method"] == "reduced-G").any()
    lines = []
    lines.append(r"\begin{table}[H]")
    lines.append(r"\centering")
    lines.append(r"\small")
    lines.append(r"\caption{Gaussian experiments with the Section 5.1 universal-inference (UI) studentized detector. Panel A gives point-null medians with median$/\{B/I_t\}$ in parentheses. Panel B gives the all-start UI detector $D_t^{t,w}=\inf_\theta\sum_s w_s R_{s:t}^{\theta}$; $d_{\rm UI,FT}$ solves the finite-prefix UI log-sum information-balance equation.}")
    lines.append(r"\label{tab:gaussian-ui-compact}")
    lines.append(r"\begin{tabular}{rcc}" if include_reduced else r"\begin{tabular}{rc}")
    lines.append(r"\toprule")
    if include_reduced:
        lines.append(r"\multicolumn{3}{c}{Panel A: point-null threshold convergence}\\")
        lines.append(r"\midrule")
        lines.append(r"$A$ & UI $t$ & reduced $G$\\")
    else:
        lines.append(r"\multicolumn{2}{c}{Panel A: point-null threshold convergence}\\")
        lines.append(r"\midrule")
        lines.append(r"$A$ & UI $t$\\")
    lines.append(r"\midrule")

    if not point_df.empty:
        for A in POINT_LEVELS:
            ui_row = point_df[(point_df["A"] == A) & (point_df["method"] == "UI-t")].iloc[0]
            ui_cell = cell_median_ratio(ui_row["median"], ui_row["ratio"])
            if include_reduced:
                g_row = point_df[(point_df["A"] == A) & (point_df["method"] == "reduced-G")].iloc[0]
                g_cell = cell_median_ratio(g_row["median"], g_row["ratio"])
                lines.append(rf"{fmt_A_latex(A)} & {ui_cell} & {g_cell}\\")
            else:
                lines.append(rf"{fmt_A_latex(A)} & {ui_cell}\\")

    col_count = 3 if include_reduced else 2
    lines.append(r"\midrule")
    lines.append(rf"\multicolumn{{{col_count}}}{{c}}{{Panel B: composite Section 5.1 UI detector}}\\")
    lines.append(r"\midrule")
    if include_reduced:
        lines.append(r"control; $T/d_0$ & $(d_0,d_{\rm UI,FT})$ & median/$d_{\rm UI,FT}$\\")
    else:
        lines.append(r"control; $T/d_0$ & $(d_0,d_{\rm UI,FT})$; median/$d_{\rm UI,FT}$\\")
    lines.append(r"\midrule")

    if not composite_df.empty:
        for _, row in composite_df.iterrows():
            label = rf"{row['control']}; ${int(row['T_over_d0'])}$"
            pair = rf"$({row['d0']:.1f},{row['d_ui_ft']:.1f})$"
            med = rf"${fmt_num(row['median'])}/{row['d_ui_ft']:.1f}={row['ratio_finite']:.3f}$"
            if include_reduced:
                lines.append(rf"{label} & {pair} & {med}\\")
            else:
                lines.append(rf"{label} & {pair}; {med}\\")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")
    return "\n".join(lines)


latex_table = make_latex_table(point_df, composite_df)
print(latex_table)

Path("gaussian_ui_point_null.csv").write_text(point_df.to_csv(index=False))
Path("gaussian_ui_composite.csv").write_text(composite_df.to_csv(index=False))
Path("gaussian_ui_table.tex").write_text(latex_table)
print("\nSaved: gaussian_ui_point_null.csv, gaussian_ui_composite.csv, gaussian_ui_table.tex")